In [1]:
import os
import json
import requests
from collections import Counter
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

current_dir = os.getcwd()  # in notebooks we use getcwd() instead of __file__
print(f"Working directory: {current_dir}")

Working directory: c:\Users\tariq\CSKG-Graph_RAG\Stage_1\OpenAlex_Enrichment


In [2]:
papers_file = os.path.join(current_dir, "openalex_papers.json")

with open(papers_file, "r", encoding="utf-8") as f:
    papers = json.load(f)

print(f"Total papers loaded: {len(papers)}")
print(f"\nFirst paper:")
print(json.dumps(papers[0], indent=2))

Total papers loaded: 10

First paper:
{
  "openalex_id": "https://openalex.org/W3200264958",
  "doi": "https://doi.org/10.1007/s12652-021-03488-z",
  "title": "Transfer learning for image classification using VGG19: Caltech-101 image data set",
  "year": 2021,
  "citations": 356,
  "authors": [
    "Monika Bansal",
    "Munish Kumar",
    "Monika Sachdeva"
  ],
  "facts_in_cskg": 7
}


In [3]:
seen_titles = set() #a set removes duplication by definition
unique_papers = []
for paper in papers:
    title = paper["title"]
    if title not in seen_titles:
        seen_titles.add(title)
        unique_papers.append(paper)

print(f"Before dedup: {len(papers)} papers")
print(f"After dedup:  {len(unique_papers)} papers")
print(f"\nUnique titles:")
for p in unique_papers:
    print(f"  - {p['title'][:60]}")



Before dedup: 10 papers
After dedup:  8 papers

Unique titles:
  - Transfer learning for image classification using VGG19: Calt
  - Robust Image Sentiment Analysis Using Progressively Trained 
  - Small-Sample Image Classification Method of Combining Protot
  - Genetic Programming With a New Representation to Automatical
  - Image Classification Using Transfer Learning and Deep Learni
  - Art Classification with Pytorch Using Transfer Learning
  - Effects of Image Degradation and Degradation Removal to CNN-
  - Evolutionary Deep Learning: A Genetic Programming Approach t


In [8]:
POLITE_EMAIL = "Tariq.Ribhi.Yaseen@gmail.com"

# Take the first paper
paper = unique_papers[1]
openalex_id = paper["openalex_id"]
short_id = openalex_id.split("/")[-1]#split returns a list, ['https:', '', 'openalex.org', 'W3200264958'], [-1] takes the last element which is the ID

print(f"Fetching: {paper['title']}")
print(f"OpenAlex ID: {short_id}")

url = f"https://api.openalex.org/works/{short_id}?mailto={POLITE_EMAIL}"
response = requests.get(url, timeout=10)
data = response.json()#data is a json object (dictionary) containing all the information about the paper
print(data)
print("\n\n\n\n")

print(f"\nFields returned by OpenAlex:")
for key in data.keys():
    print(f"  - {key}")

Fetching: Robust Image Sentiment Analysis Using Progressively Trained and Domain Transferred Deep Networks
OpenAlex ID: W2253891449
{'id': 'https://openalex.org/W2253891449', 'doi': 'https://doi.org/10.48550/arxiv.1509.06041', 'title': 'Robust Image Sentiment Analysis Using Progressively Trained and Domain Transferred Deep Networks', 'display_name': 'Robust Image Sentiment Analysis Using Progressively Trained and Domain Transferred Deep Networks', 'publication_year': 2015, 'publication_date': '2015-09-20', 'ids': {'openalex': 'https://openalex.org/W2253891449', 'doi': 'https://doi.org/10.48550/arxiv.1509.06041', 'mag': '2253891449'}, 'language': 'en', 'primary_location': {'id': 'pmh:oai:arXiv.org:1509.06041', 'is_oa': True, 'landing_page_url': 'http://arxiv.org/abs/1509.06041', 'pdf_url': 'https://arxiv.org/pdf/1509.06041', 'source': {'id': 'https://openalex.org/S4306400194', 'display_name': 'arXiv (Cornell University)', 'issn_l': None, 'issn': None, 'is_oa': True, 'is_in_doaj': False,

In [9]:
abstract_index = data.get("abstract_inverted_index", {})#get is a method of dictionaries that returns the value for the given key, if the key is not found it returns the default value which is an empty dictionary in this case

if not abstract_index:
    print("No abstract inverted index found for this paper.")
else:
    print("Raw inverted index (first 10 entries):")
    for i, (word, positions) in enumerate(abstract_index.items()):
        print(f"  '{word}' → positions {positions}")
        if i >= 9:
            print("  ...")
            break

Raw inverted index (first 10 entries):
  'Sentiment' → positions [0, 54]
  'analysis' → positions [1, 22, 55, 212]
  'of' → positions [2, 56, 81, 110, 151, 183]
  'online' → positions [3]
  'user' → positions [4, 66]
  'generated' → positions [5]
  'content' → positions [6, 61, 85]
  'is' → positions [7, 86]
  'important' → positions [8]
  'for' → positions [9, 127]
  ...


In [ ]:
def reconstruct_abstract(inverted_index: dict) -> str:
    if not inverted_index:
        return ""
    # Step 1: flip (word, [positions]) → [(pos, word), (pos, word)...]
    words = []
    for word, positions in inverted_index.items():#the inverted index is a dictionary where the keys are words and the values are lists of positions where those words appear in the abstract. We iterate over each word and its corresponding positions.
        for pos in positions:
            words.append((pos, word))
    # Step 2: sort by position number
    words.sort(key=lambda x: x[0])
    # Step 3: join just the words in order
    return " ".join(word for _, word in words)

abstract = reconstruct_abstract(abstract_index)
print("Reconstructed abstract:")
print(abstract)

Reconstructed abstract:
Sentiment analysis of online user generated content is important for many social media analytics tasks. Researchers have largely relied on textual sentiment analysis to develop systems to predict political elections, measure economic indicators, and so on. Recently, social media users are increasingly using images and videos to express their opinions and share their experiences. Sentiment analysis of such large scale visual content can help better extract user sentiments toward events or topics, such as those in image tweets, so that prediction of sentiment from visual content is complementary to textual sentiment analysis. Motivated by the needs in leveraging large scale yet noisy training data to solve the extremely challenging problem of image sentiment analysis, we employ Convolutional Neural Networks (CNN). We first design a suitable CNN architecture for image sentiment analysis. We obtain half a million training samples by using a baseline sentiment algori